In [0]:
CREATE OR REFRESH STREAMING TABLE dataflatform_dev.silver.user_info_cleaned
COMMENT "Streaming table containing cleaned and transformed user data"
TBLPROPERTIES("table.layer"="silver", "table.type"="transformation")
AS
select _databricks_row_hash,
pseudo_user_id,
stream_id,
user_info.first_purchase_date,
user_info.user_first_touch_timestamp_micros
from STREAM(`dataflatform_dev`.`bronze`.`pseudonymous_users`)


In [0]:
CREATE OR REFRESH STREAMING TABLE dataflatform_dev.silver.user_info_upserted
COMMENT "Streaming table containing latest users data."
TBLPROPERTIES("table.layer"="silver", "table.type"="transformation as scd-type1");
CREATE FLOW silver_user_info_upserted
AS AUTO CDC INTO silver.user_info_upserted
FROM STREAM(silver.user_info_cleaned)
KEYS (pseudo_user_id)
SEQUENCE BY first_purchase_date
COLUMNS * EXCEPT (stream_id)
STORED AS SCD TYPE 1;


In [0]:
CREATE OR REFRESH STREAMING TABLE dataflatform_dev.silver.user_device_cleaned
COMMENT "Streaming table containing cleaned and transformed user device data"
TBLPROPERTIES("table.layer"="silver")
AS
select _databricks_row_hash,
pseudo_user_id,
stream_id,
last_updated_date,
device.operating_system,
device.category,
device.mobile_model_name,
device.mobile_brand_name
from STREAM(`dataflatform_dev`.`bronze`.`pseudonymous_users`)  

In [0]:
CREATE OR REFRESH STREAMING TABLE dataflatform_dev.silver.user_device_upserted
COMMENT "Streaming table containing latest users data."
TBLPROPERTIES("table.layer"="silver", "table.type"="transformation as scd-type1");
CREATE FLOW silver_user_device_upserted
AS AUTO CDC INTO silver.user_device_upserted
FROM STREAM(silver.user_device_cleaned)
KEYS (pseudo_user_id)
SEQUENCE BY last_updated_date
COLUMNS * EXCEPT (stream_id,last_updated_date)
STORED AS SCD TYPE 1;
